In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, StructType, StructField, IntegerType, TimestampType, FloatType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

In [0]:
df_bronze=spark.table("crypto.bronze.brz_crypto")

display(df_bronze.limit(10))



In [0]:
 print(f"Row count: {df_bronze.count()}")
 print(f"Column count: {len(df_bronze.columns)}")

In [0]:
df_bronze.printSchema()


## Check for nulls

In [0]:
nulls=df_bronze.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in df_bronze.columns])
display(nulls)

In [0]:
df_silver=df_bronze.select(
    col("id"),
    col("symbol"),
    col("name"),
    col("current_price"),
    col("market_cap"),
    col("total_volume"),
    col("price_change_percentage_24h")
)

display(df_silver.limit(10))


In [0]:
nulls=df_silver.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in df_silver.columns])
display(nulls)

In [0]:
df_silver.select("symbol").distinct().display()

Check for duplicates

In [0]:
df_silver.groupBy("id").count().filter("count(*) > 1").show()

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("crypto.silver.slv_crypto")

In [0]:
%sql
SELECT * FROM crypto.silver.slv_crypto
LIMIT 10